<a href="https://colab.research.google.com/github/cylaadhan/FuzzyLogic/blob/main/indoBERT1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ============================================================
# FULL CODE MAPPING DATA MATRIK KE BEBERAPA DATA STANDAR
# MENGGUNAKAN IndoBERT + COSINE SIMILARITY
# ============================================================

# 1. INSTALL LIBRARY
!pip install transformers torch pandas openpyxl scikit-learn tqdm -q

# 2. IMPORT LIBRARY
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
from google.colab import files

# 4. NAMA FILE
# Sesuaikan kalau nama file di Colab berbeda
file_standar = "standar1.xlsx"
file_matrik = "matrik.xlsx"

# 5. BACA FILE EXCEL
# Kalau sheet berbeda, nanti akan otomatis coba sheet pertama
try:
    df_standar = pd.read_excel(file_standar, sheet_name="Data_Bersih")
except:
    df_standar = pd.read_excel(file_standar)

try:
    df_matrik = pd.read_excel(file_matrik, sheet_name="Sheet1")
except:
    df_matrik = pd.read_excel(file_matrik)

print("Jumlah data standar:", df_standar.shape)
print("Jumlah data matrik:", df_matrik.shape)

print("\nKolom standar:")
print(df_standar.columns.tolist())

print("\nKolom matrik:")
print(df_matrik.columns.tolist())

# 6. CEK KOLOM FULL
if "FULL" not in df_standar.columns:
    raise ValueError("Kolom FULL tidak ditemukan di file standar")

if "FULL" not in df_matrik.columns:
    raise ValueError("Kolom FULL tidak ditemukan di file matrik")

# 7. BERSIHKAN DATA TEKS
df_standar["FULL"] = df_standar["FULL"].astype(str).fillna("")
df_matrik["FULL"] = df_matrik["FULL"].astype(str).fillna("")

teks_standar = df_standar["FULL"].tolist()
teks_matrik = df_matrik["FULL"].tolist()

# 8. LOAD MODEL IndoBERT
model_name = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("\nModel IndoBERT berhasil dimuat")
print("Device yang digunakan:", device)

# 9. FUNGSI MEAN POOLING
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    embeddings = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(
        input_mask_expanded.sum(1),
        min=1e-9
    )

    return embeddings

# 10. FUNGSI MEMBUAT EMBEDDING
def get_embeddings(texts, batch_size=16, max_length=256):
    embeddings = []
    model.eval()

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[i:i+batch_size]

            encoded_input = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            encoded_input = {
                key: value.to(device)
                for key, value in encoded_input.items()
            }

            model_output = model(**encoded_input)

            batch_embeddings = mean_pooling(
                model_output,
                encoded_input["attention_mask"]
            )

            embeddings.append(batch_embeddings.cpu().numpy())

    return np.vstack(embeddings)

# 11. BUAT EMBEDDING
print("\nMembuat embedding data matrik...")
embedding_matrik = get_embeddings(teks_matrik, batch_size=16)

print("\nMembuat embedding data standar...")
embedding_standar = get_embeddings(teks_standar, batch_size=16)

print("\nUkuran embedding matrik:", embedding_matrik.shape)
print("Ukuran embedding standar:", embedding_standar.shape)

# 12. HITUNG SIMILARITY
similarity_matrix = cosine_similarity(embedding_matrik, embedding_standar)

print("\nUkuran similarity matrix:", similarity_matrix.shape)

# 13. SETTING MAPPING
# TOP_K = jumlah maksimal standar yang dicari untuk 1 data matrik
# THRESHOLD = batas minimal kemiripan
TOP_K = 5
THRESHOLD = 0.70

# 14. PROSES MAPPING ONE-TO-MANY
hasil_mapping = []

for idx_matrik in range(len(df_matrik)):
    similarities = similarity_matrix[idx_matrik]

    top_indices = similarities.argsort()[::-1][:TOP_K]

    for rank, idx_standar in enumerate(top_indices, start=1):
        score = similarities[idx_standar]

        if score >= THRESHOLD:
            data = {
                "NO_MATRIK": idx_matrik + 1,
                "RANK": rank,
                "FULL_MATRIK": df_matrik.loc[idx_matrik, "FULL"],

                "NO_STANDAR": idx_standar + 1,
                "FULL_STANDAR": df_standar.loc[idx_standar, "FULL"],

                "SIMILARITY": round(float(score), 4)
            }

            # Tambahkan kolom dari file matrik jika ada
            for col in df_matrik.columns:
                if col != "FULL":
                    data[f"MATRIK_{col}"] = df_matrik.loc[idx_matrik, col]

            # Tambahkan kolom dari file standar jika ada
            for col in df_standar.columns:
                if col != "FULL":
                    data[f"STANDAR_{col}"] = df_standar.loc[idx_standar, col]

            hasil_mapping.append(data)

df_hasil = pd.DataFrame(hasil_mapping)

print("\nJumlah hasil mapping dengan threshold:", len(df_hasil))

# 15. BUAT HASIL TOP 10 TANPA THRESHOLD
hasil_top10 = []

TOP_K_SEMUA = 10

for idx_matrik in range(len(df_matrik)):
    similarities = similarity_matrix[idx_matrik]
    top_indices = similarities.argsort()[::-1][:TOP_K_SEMUA]

    for rank, idx_standar in enumerate(top_indices, start=1):
        score = similarities[idx_standar]

        data = {
            "NO_MATRIK": idx_matrik + 1,
            "RANK": rank,
            "FULL_MATRIK": df_matrik.loc[idx_matrik, "FULL"],

            "NO_STANDAR": idx_standar + 1,
            "FULL_STANDAR": df_standar.loc[idx_standar, "FULL"],

            "SIMILARITY": round(float(score), 4)
        }

        for col in df_matrik.columns:
            if col != "FULL":
                data[f"MATRIK_{col}"] = df_matrik.loc[idx_matrik, col]

        for col in df_standar.columns:
            if col != "FULL":
                data[f"STANDAR_{col}"] = df_standar.loc[idx_standar, col]

        hasil_top10.append(data)

df_top10 = pd.DataFrame(hasil_top10)

print("Jumlah hasil top 10:", len(df_top10))

# 16. SIMPAN KE EXCEL DALAM 1 FILE, 2 SHEET
output_file = "hasil_mapping_indobert.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_hasil.to_excel(writer, sheet_name="Mapping_Threshold", index=False)
    df_top10.to_excel(writer, sheet_name="Top10_Semua", index=False)

print("\nFile berhasil dibuat:", output_file)

# 17. DOWNLOAD FILE
files.download(output_file)

Jumlah data standar: (357, 4)
Jumlah data matrik: (75, 4)

Kolom standar:
['STANDAR', 'PERNYATAAN ISI STANDAR', 'INDIKATOR', 'FULL']

Kolom matrik:
['KRITERIA', 'ELEMEN', 'DESKRIPTOR', 'FULL']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...


Membuat embedding data standar...


100%|██████████| 23/23 [01:18<00:00,  3.41s/it]



Ukuran embedding matrik: (75, 768)
Ukuran embedding standar: (357, 768)

Ukuran similarity matrix: (75, 357)

Jumlah hasil mapping dengan threshold: 375
Jumlah hasil top 10: 750

File berhasil dibuat: hasil_mapping_indobert.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>